# Mesa de Bar — Análise da Simulação Multi‑agente

Notebook de análise **ponta a ponta** das sessões geradas pelo módulo de simulação
(`data/sim_runs/<run_id>/`). Lê a árvore de sessões, monta as tabelas *tidy*,
calcula métricas semânticas e de engenharia, roda os modelos e produz figuras,
tabelas e uma análise qualitativa.

> **Enquadramento.** As alegações são sobre o **coletivo de agentes e a arquitetura
> do loop** — não sobre humanos. "Auto‑preferência de fonte" aqui é sensibilidade
> à autoria de avaliadores LLM (juiz/jogador‑agente).

## Funcionalidades e análises implementadas

**A. Ingestão & integridade**
1. Parser da árvore `sim_runs/<run_id>/sessions/*` → tabelas `sessions`, `ideas`, `rounds`, `calls`.
2. Reconstrução de autoria (player_id → papel: jogador‑agente / persona / dono).
3. Flags por ideia: escolhida‑pelo‑juiz, escolhida‑pelo‑jogador, **avançou**.
4. Conferência manifesto‑vs‑configuração, balanceamento por célula/tema/jogador, `parse_status`, falhas.

**B. Validação de engenharia (quantitativo de tokens/custo/números)**
5. Tokens (in/out) por papel, rodada, sessão e célula.
6. **Custo** estimado (preço configurável por modelo) por papel/sessão/célula/total.
7. Latência por papel; nº de chamadas; taxa de falha/contingência. Tabela de confiabilidade (**T7**).
8. *Flag* de instrumentação: chamadas do **jogador‑agente** não aparecem no `llm_calls.jsonl` (subcontagem).

**C. Métricas semânticas (embeddings)**
9. Embedding multilíngue de todas as ideias (fallback TF‑IDF), com cache por `idea_id`.
10. **Originalidade** (distância ao *reference set*), **diversidade de pool** (intra‑sessão),
    **diversidade coletiva** (entre sessões — DV central), **flexibilidade** (nº de clusters).

**D. Dinâmica do loop**
11. **Homogeneização por dinâmica** `best_idea` (RQ2 / figura‑manchete **F2**).
12. **Trajetória de convergência** por rodada e *slope* (RQ4 / **F3**).
13. **Concordância juiz–jogador** ao longo das rodadas (**F6**).
14. **Autoria dos vencedores** por papel e por dinâmica.

**E. Sensibilidade à fonte & síntese**
15. Índice de **auto‑preferência** por escolhedor (RQ3 — quando a cegueira varia).
16. Efeito da **síntese** (originalidade/diversidade da síntese vs pool).

**F. Estatística**
17. Modelos (LMM/GLMM) com *guards* para N pequeno; efeitos + IC; nota de poder.

**G. Análise qualitativa de conteúdo**
18. Evolução das ideias (cadeia de vencedores), padrões nas justificativas do juiz,
    comportamento das personas, exemplos lado a lado, mapa 2D dos embeddings (**F7**).

**H. Insights & exportações**
19. Sumário executivo automático (números) + leitura.
20. Exportação das tabelas *tidy* (CSV) e das figuras (PNG).


## 0. Configuração

In [1]:
import os, glob, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# >>> AJUSTE AQUI <<<
# Caminho do run a analisar (a pasta que contém run_manifest.json e sessions/).
DATA_ROOT = os.environ.get("MDB_RUN_DIR", "../data/sim_runs/sim_20260701_130722")
OUT = "analise_saidas"                      # onde salvar CSVs e figuras
os.makedirs(f"{OUT}/figuras", exist_ok=True)
os.makedirs(f"{OUT}/tabelas", exist_ok=True)

# Preços por 1k tokens (in, out). CONFIRME os valores vigentes na sua conta OpenAI.
PRICES = {
    "gpt-4o-mini":  (0.00015, 0.00060),
    "gpt-4o":       (0.00250, 0.01000),
    "gpt-4.1-mini": (0.00040, 0.00160),
}
def call_cost(model, t_in, t_out):
    pi, po = PRICES.get(model, (0.0, 0.0)); return t_in/1000*pi + t_out/1000*po

# Embedding: multilíngue (ideal) com fallback TF-IDF se a lib/rede não estiver disponível.
EMBED_MODEL = "paraphrase-multilingual-mpnet-base-v2"

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid", context="notebook")
except Exception: pass
plt.rcParams["figure.dpi"] = 110
print("DATA_ROOT =", DATA_ROOT)
print("existe?   ", os.path.isdir(DATA_ROOT))

DATA_ROOT = ../data/sim_runs/sim_20260701_130722
existe?    True


## 1. Ingestão e parsing

Lê cada pasta de sessão e monta quatro tabelas *tidy* chaveadas por ids. A
estrutura de eventos é uniforme entre os modos de `best_idea` — sempre há
`round.choices_compared` (resolve os picks do jogador e do juiz para `player_id`)
e `round.idea_advanced` (o vencedor), então um único parser serve aos três modos.

In [2]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

def player_id_maps(manifest):
    """player_id -> (role, name): 0=owner, 1=jogador-agente, 2..=personas."""
    role = {0:"owner", 1:"player"}; name = {0:"Dono do Bar", 1:manifest.get("user_name","Jogador")}
    for i, p in enumerate(manifest.get("personas", [])):
        role[i+2] = "persona"; name[i+2] = p
    return role, name

def cell_fields(cell, man):
    g = lambda k, d=None: cell.get(k, man.get(k, d))
    return dict(
        cell_best_idea=g("best_idea"), cell_composicao=g("composicao"),
        cell_tema=g("tema", man.get("topic")), cell_source_blind=g("source_blind_on_choice"),
        cell_judge_criterion=g("judge_criterion"),
        cell_model_pairing=g("model_pairing", man.get("model_pairing")), cell_rounds=g("rounds"))

def cell_id_from_factors(row):
    return "|".join(f"{k.replace('cell_','')}={row.get(k)}" for k in
        ["cell_best_idea","cell_composicao","cell_source_blind","cell_model_pairing","cell_judge_criterion"])

def parse_session(sess_dir):
    man = json.load(open(os.path.join(sess_dir,"manifest.json"), encoding="utf-8"))
    evs = read_jsonl(os.path.join(sess_dir,"events.jsonl"))
    calls = read_jsonl(os.path.join(sess_dir,"llm_calls.jsonl"))
    try: final = json.load(open(os.path.join(sess_dir,"final_idea.json"), encoding="utf-8"))
    except Exception: final = {}
    sid = man["session_id"]; role, name = player_id_maps(man); cell = man.get("sim_cell", {})
    cmp_by_r = {e["round"]: e["payload"] for e in evs if e["type"]=="round.choices_compared"}
    adv_by_r = {e["round"]: e["payload"] for e in evs if e["type"]=="round.idea_advanced"}
    pj_by_r  = {e["round"]: e["payload"] for e in evs if e["type"]=="player.judged"}

    ideas = []
    for e in evs:
        if e["type"] != "round.proposal_submitted": continue
        r, pid = e["round"], e["payload"]["player_id"]; c = cmp_by_r.get(r,{}); a = adv_by_r.get(r,{})
        txt = e["payload"]["idea_text"]
        ideas.append(dict(idea_id=f"{sid}_r{r}_p{pid}", session_id=sid, round=r, player_id=pid,
            author_role=role.get(pid,"?"), author_name=name.get(pid,"?"), text=txt,
            idea_chars=e["payload"].get("idea_chars", len(txt)),
            selected_by_player=(c.get("player_pick_id")==pid),
            selected_by_judge=(c.get("judge_pick_id")==pid),
            advanced=(a.get("advanced_player_id")==pid), **cell_fields(cell,man)))

    rounds = []
    for r in sorted({e["round"] for e in evs if e.get("round")}):
        c = cmp_by_r.get(r,{}); a = adv_by_r.get(r,{}); pj = pj_by_r.get(r,{})
        if not c and not a: continue
        rounds.append(dict(session_id=sid, round=r, player_pick_id=c.get("player_pick_id"),
            judge_pick_id=c.get("judge_pick_id"), match=c.get("match"), agree_judge=pj.get("agree_judge"),
            advanced_player_id=a.get("advanced_player_id"),
            advanced_role=role.get(a.get("advanced_player_id")), rule=a.get("rule"), **cell_fields(cell,man)))

    crows = []
    for c in calls:
        ctx = c.get("context",{}); tok = c.get("tokens",{}); ti,to = tok.get("in",0),tok.get("out",0)
        crows.append(dict(session_id=sid, call_id=c.get("call_id"), actor_role=ctx.get("actor_role"),
            actor_id=ctx.get("actor_id"), round=ctx.get("round"), prompt_id=c.get("prompt_id"),
            provider=c.get("provider"), model=c.get("model"), tokens_in=ti, tokens_out=to,
            latency_ms=c.get("latency_ms"), parse_status=c.get("parse_status"),
            error=bool(c.get("error")), cost_usd=call_cost(c.get("model"),ti,to), **cell_fields(cell,man)))

    winners = [a.get("advanced_player_id") for a in adv_by_r.values()]
    srow = dict(session_id=sid, run_id=man.get("sim_run_id"), player_persona=man.get("player_persona"),
        player_temperature=man.get("player_temperature"), topic=man.get("topic"),
        n_rounds=man.get("rounds"), status=man.get("status"),
        provider=man.get("llm",{}).get("provider"), model=man.get("llm",{}).get("model"),
        seed=man.get("seed"), simulated=man.get("simulated", False), n_ideas=len(ideas), n_calls=len(calls),
        tokens_in=sum(x["tokens_in"] for x in crows), tokens_out=sum(x["tokens_out"] for x in crows),
        cost_usd=sum(x["cost_usd"] for x in crows),
        latency_ms_total=sum((x["latency_ms"] or 0) for x in crows),
        n_failures=sum(1 for x in crows if x["error"]),
        winners_player=sum(1 for w in winners if role.get(w)=="player"),
        winners_persona=sum(1 for w in winners if role.get(w)=="persona"),
        match_rate=np.mean([1 if r["match"] else 0 for r in rounds]) if rounds else np.nan,
        final_summary=final.get("summary"), synth_chars=len(final.get("summary") or ""), **cell_fields(cell,man))
    return srow, ideas, rounds, crows

def load_run(run_dir):
    S,I,R,C = [],[],[],[]
    for sd in sorted(glob.glob(os.path.join(run_dir,"sessions","*"))):
        if not os.path.isfile(os.path.join(sd,"manifest.json")): continue
        srow, ideas, rounds, crows = parse_session(sd)
        S.append(srow); I+=ideas; R+=rounds; C+=crows
    sessions, ideas = pd.DataFrame(S), pd.DataFrame(I)
    rounds, calls = pd.DataFrame(R), pd.DataFrame(C)
    for df in (sessions, ideas, rounds):
        if len(df): df["cell_id"] = df.apply(cell_id_from_factors, axis=1)
    return dict(sessions=sessions, ideas=ideas, rounds=rounds, calls=calls)

In [3]:
D = load_run(DATA_ROOT)
sessions, ideas, rounds, calls = D["sessions"], D["ideas"], D["rounds"], D["calls"]
print("Tabelas carregadas:")
for k,v in D.items(): print(f"  {k:9s}: {v.shape[0]:4d} linhas × {v.shape[1]} colunas")
display(sessions.head())
display(ideas.head())

Tabelas carregadas:
  sessions :  108 linhas × 31 colunas
  ideas    : 1620 linhas × 19 colunas
  rounds   :  324 linhas × 17 colunas
  calls    : 3672 linhas × 21 colunas


,session_id,run_id,player_persona,player_temperature,topic,n_rounds,status,provider,model,seed,...,final_summary,synth_chars,cell_best_idea,cell_composicao,cell_tema,cell_source_blind,cell_judge_criterion,cell_model_pairing,cell_rounds,cell_id
0,20260701-130722_41ed15ee,sim_20260701_130722,cauteloso_t07,0.7,Como reduzir o desperdício de comida em casa?,3,summary_ready,openai,gpt-4o-mini,1278491942,...,# Resumo Final da Ideia: Redução do Desperdíci...,2207,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
1,20260701-130823_232719bc,sim_20260701_130722,cauteloso_t07,0.7,Como reduzir o desperdício de comida em casa?,3,summary_ready,openai,gpt-4o-mini,977396083,...,# Resumo Final da Ideia: Reduzindo o Desperdíc...,1867,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
2,20260701-130905_73efa694,sim_20260701_130722,equilibrado_t07,0.7,Como reduzir o desperdício de comida em casa?,3,summary_ready,openai,gpt-4o-mini,61814544,...,# Resumo Final da Ideia: Redução do Desperdíci...,1739,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
3,20260701-130938_a5480b01,sim_20260701_130722,equilibrado_t07,0.7,Como reduzir o desperdício de comida em casa?,3,summary_ready,openai,gpt-4o-mini,2051983750,...,# Resumo Final da Ideia: Reduzindo o Desperdíc...,2321,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
4,20260701-131018_63155938,sim_20260701_130722,ousado_t07,0.7,Como reduzir o desperdício de comida em casa?,3,summary_ready,openai,gpt-4o-mini,360507124,...,# Resumo Final da Ideia: Redução do Desperdíci...,2153,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...


,idea_id,session_id,round,player_id,author_role,author_name,text,idea_chars,selected_by_player,selected_by_judge,advanced,cell_best_idea,cell_composicao,cell_tema,cell_source_blind,cell_judge_criterion,cell_model_pairing,cell_rounds,cell_id
0,20260701-130722_41ed15ee_r1_p1,20260701-130722_41ed15ee,1,1,player,Jogador Cauteloso (T=0.7),Eu posso criar um planejamento semanal de refe...,187,True,True,True,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
1,20260701-130722_41ed15ee_r1_p2,20260701-130722_41ed15ee,1,2,persona,Dona Cleide,"É verdade, aproveitar tudo na cozinha é essenc...",226,False,False,False,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
2,20260701-130722_41ed15ee_r1_p3,20260701-130722_41ed15ee,1,3,persona,Seu Antônio,"Concordo, Dono do Bar. Planejar as compras é e...",225,False,False,False,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
3,20260701-130722_41ed15ee_r1_p4,20260701-130722_41ed15ee,1,4,persona,Seu Raimundo,"João, tu tá certo em falar das sobras! Lembro ...",191,False,False,False,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...
4,20260701-130722_41ed15ee_r1_p5,20260701-130722_41ed15ee,1,5,persona,Bia,"Concordo com você, Jogador Cauteloso! Planejam...",218,False,False,False,judge,comum,Como reduzir o desperdício de comida em casa?,True,composto,same_model,3,best_idea=judge|composicao=comum|source_blind=...


## 2. Integridade e balanceamento

Confere se o que rodou bate com o que foi declarado, e mede o equilíbrio do desenho.

In [4]:
rep = {}
rep["n_sessions"] = len(sessions)
rep["rounds_ok"] = bool((sessions["n_rounds"] == rounds.groupby("session_id")["round"].max()
                         .reindex(sessions["session_id"]).values).all())
rep["ideas_por_sessao"] = ideas.groupby("session_id").size().describe().to_dict()
rep["parse_status"] = calls["parse_status"].value_counts().to_dict() if len(calls) else {}
rep["falhas"] = int(calls["error"].sum()) if len(calls) else 0

print("=== Balanceamento por célula × tema (nº de sessões) ===")
display(sessions.groupby(["cell_best_idea","cell_tema"]).size().rename("n_sessions").reset_index())
print("=== Sessões por jogador sintético ===")
display(sessions["player_persona"].value_counts().rename("n").reset_index())

# Papéis presentes nas chamadas (flag do jogador-agente não instrumentado)
roles = set(calls["actor_role"].unique()) if len(calls) else set()
print("\nPapéis em llm_calls:", sorted(roles))
if "player" not in roles:
    print("⚠️  As chamadas do JOGADOR-AGENTE não estão no llm_calls.jsonl: "
          "tokens/custo abaixo SUBCONTAM o jogador. Instrumentar para custo completo.")
rep

=== Balanceamento por célula × tema (nº de sessões) ===


,cell_best_idea,cell_tema,n_sessions
0,judge,Como reduzir as alucinações de modelos de ling...,6
1,judge,Como reduzir o desperdício de comida em casa?,6
2,judge,"Como seria uma cidade projetada para o ócio, e...",6
3,judge,Como uma cidade média pode se adaptar a evento...,6
4,judge,O que torna uma vida bem vivida?,6
5,judge,Uma nova forma de as pessoas fazerem amizades ...,6
6,player,Como reduzir as alucinações de modelos de ling...,6
7,player,Como reduzir o desperdício de comida em casa?,6
8,player,"Como seria uma cidade projetada para o ócio, e...",6
9,player,Como uma cidade média pode se adaptar a evento...,6


=== Sessões por jogador sintético ===


,player_persona,n
0,cauteloso_t07,36
1,equilibrado_t07,36
2,ousado_t07,36



Papéis em llm_calls: ['judge', 'owner', 'persona', 'persona_reflect', 'player']


{'n_sessions': 108,
 'rounds_ok': True,
 'ideas_por_sessao': {'count': 108.0,
  'mean': 15.0,
  'std': 0.0,
  'min': 15.0,
  '25%': 15.0,
  '50%': 15.0,
  '75%': 15.0,
  'max': 15.0},
 'parse_status': {'ok': 3672},
 'falhas': 0}

## 3. Validação de engenharia — tokens, custo, latência (quantitativo)

Sustenta a alegação "o aparato funciona como declarado" e dá o custo real de rodar
o desenho. Custo = tokens × preço por modelo (configurável em `PRICES`).

In [5]:
def agg_calls(df, by):
    g = df.groupby(by).agg(n_calls=("call_id","size"), tokens_in=("tokens_in","sum"),
        tokens_out=("tokens_out","sum"), latency_ms_mean=("latency_ms","mean"),
        cost_usd=("cost_usd","sum")).round(5)
    g["tokens_total"] = g["tokens_in"] + g["tokens_out"]
    return g.reset_index()

por_papel = agg_calls(calls, "actor_role")
por_celula = agg_calls(calls, "cell_best_idea")
print("=== Custo/tokens por papel (todo o run) ==="); display(por_papel)
print("=== Custo/tokens por dinâmica (best_idea) ==="); display(por_celula)

# Tabela de confiabilidade do aparato (T7)
T7 = pd.DataFrame({
    "metrica": ["sessões","chamadas LLM","tokens in","tokens out","tokens total",
                "custo total (US$)","custo médio/sessão (US$)","latência média/chamada (ms)",
                "taxa de falha","parse ok (%)"],
    "valor": [len(sessions), len(calls), int(calls["tokens_in"].sum()), int(calls["tokens_out"].sum()),
              int(calls["tokens_in"].sum()+calls["tokens_out"].sum()), round(calls["cost_usd"].sum(),4),
              round(calls["cost_usd"].sum()/max(len(sessions),1),5), round(calls["latency_ms"].mean(),0),
              round(calls["error"].mean(),4),
              round(100*(calls["parse_status"].eq("ok").mean() if len(calls) else 0),1)]})
display(T7); T7.to_csv(f"{OUT}/tabelas/T7_confiabilidade.csv", index=False)

=== Custo/tokens por papel (todo o run) ===


,actor_role,n_calls,tokens_in,tokens_out,latency_ms_mean,cost_usd,tokens_total
0,judge,324,136973,324,1153.48457,0.02074,137297
1,owner,108,36174,49399,7961.22222,0.03507,85573
2,persona,1296,852406,102303,2215.49306,0.18924,954709
3,persona_reflect,1296,547095,58159,1936.28164,0.11696,605254
4,player,648,155574,20404,1408.76080,0.03558,175978


=== Custo/tokens por dinâmica (best_idea) ===


,cell_best_idea,n_calls,tokens_in,tokens_out,latency_ms_mean,cost_usd,tokens_total
0,judge,1224,572140,76985,2224.27614,0.13201,649125
1,player,1224,579278,76895,1779.46895,0.13303,656173
2,player_decides,1224,576804,76709,2145.86111,0.13255,653513


,metrica,valor
0,sessões,1.080000e+02
1,chamadas LLM,3.672000e+03
2,tokens in,1.728222e+06
3,tokens out,2.305890e+05
4,tokens total,1.958811e+06
5,custo total (US$),3.976000e-01
6,custo médio/sessão (US$),3.680000e-03
7,latência média/chamada (ms),2.050000e+03
8,taxa de falha,0.000000e+00
9,parse ok (%),1.000000e+02


In [6]:
# Figuras de engenharia
fig, ax = plt.subplots(1,3, figsize=(15,4))
por_papel.set_index("actor_role")[["tokens_in","tokens_out"]].plot.bar(stacked=True, ax=ax[0])
ax[0].set_title("Tokens por papel"); ax[0].set_xlabel(""); ax[0].tick_params(axis="x", rotation=30)
por_papel.set_index("actor_role")["cost_usd"].plot.bar(ax=ax[1], color="#c0504d")
ax[1].set_title("Custo (US$) por papel"); ax[1].set_xlabel("")
if calls["latency_ms"].notna().any():
    calls.boxplot(column="latency_ms", by="actor_role", ax=ax[2]); ax[2].set_title("Latência por papel")
    ax[2].set_xlabel(""); plt.suptitle("")
plt.tight_layout(); plt.savefig(f"{OUT}/figuras/eng_tokens_custo_latencia.png", bbox_inches="tight")
plt.show()

## 4. Embeddings das ideias

Tenta `sentence-transformers` multilíngue; se indisponível (lib ou rede), cai para
TF‑IDF — as métricas continuam comparáveis dentro do run. **Para o estudo final,
fixe o modelo multilíngue** e recompute como robustez.

In [7]:
def embed_texts(texts, model_name=EMBED_MODEL):
    texts = [t if isinstance(t,str) else "" for t in texts]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(model_name)
        return np.asarray(m.encode(texts, normalize_embeddings=True, show_progress_bar=False)), f"st:{model_name}"
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.preprocessing import normalize
        X = TfidfVectorizer(max_features=4096).fit_transform(texts)
        return normalize(X).toarray().astype(np.float32), "tfidf-fallback"

ideas = ideas.reset_index(drop=True)
# Embedda ideias E sínteses na MESMA chamada -> mesmo espaço vetorial (essencial p/ TF-IDF).
syn_map = sessions.set_index("session_id")["final_summary"].to_dict()
syn_texts = [syn_map.get(s) if isinstance(syn_map.get(s), str) else "" for s in sessions["session_id"]]
ALL, EMB_BACKEND = embed_texts(ideas["text"].tolist() + syn_texts)
n_ideas = len(ideas)
E = ALL[:n_ideas]
EMB = {iid: E[i] for i, iid in enumerate(ideas["idea_id"])}              # cache ideias
SYN_EMB = {sid: ALL[n_ideas+i] for i, sid in enumerate(sessions["session_id"])}  # cache sínteses
print("backend de embedding:", EMB_BACKEND, "| ideias:", E.shape, "| sínteses:", len(SYN_EMB))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4395.39it/s]


backend de embedding: st:paraphrase-multilingual-mpnet-base-v2 | ideias: (1620, 768) | sínteses: 108


## 5. Métricas semânticas

`originalidade` (distância ao *reference set*), `diversidade de pool` (intra‑sessão),
`diversidade coletiva` (entre sessões — DV central da RQ2) e `flexibilidade`
(nº de clusters). Sem a condição B0 no run, a originalidade usa um *reference set*
substituto (todas as ideias da 1ª rodada do mesmo tema); no estudo final, troque por B0.

In [8]:
def cos_matrix(M):
    M = np.asarray(M, float); n = np.linalg.norm(M,axis=1,keepdims=True); n[n==0]=1
    U = M/n; return U @ U.T
def pool_diversity(M):
    if len(M) < 2: return np.nan
    S = cos_matrix(M); iu = np.triu_indices(len(M), 1); return float(np.mean(1 - S[iu]))
def originality_vs_ref(M, R):
    if len(R)==0: return np.full(len(M), np.nan)
    A = np.asarray(M,float); B = np.asarray(R,float)
    A = A/np.clip(np.linalg.norm(A,axis=1,keepdims=True),1e-9,None)
    B = B/np.clip(np.linalg.norm(B,axis=1,keepdims=True),1e-9,None)
    return 1 - (A @ B.T).mean(axis=1)
def flexibility(M, max_k=8):
    from sklearn.cluster import KMeans; from sklearn.metrics import silhouette_score
    n=len(M)
    if n<3: return 1
    best_k,best_s=1,-1
    for k in range(2, min(max_k,n-1)+1):
        try:
            lab=KMeans(k, n_init=5, random_state=0).fit_predict(M); s=silhouette_score(M,lab)
            if s>best_s: best_s,best_k=s,k
        except Exception: pass
    return best_k

# reference set por tema = ideias da 1ª rodada (proxy de B0 enquanto não há B0)
ref_by_theme = {}
for tema, g in ideas[ideas["round"]==ideas["round"].min()].groupby("cell_tema"):
    ref_by_theme[tema] = np.array([EMB[i] for i in g["idea_id"]])

ideas["orig_score"] = np.nan
for tema, g in ideas.groupby("cell_tema"):
    M = np.array([EMB[i] for i in g["idea_id"]]); R = ref_by_theme.get(tema, M)
    ideas.loc[g.index, "orig_score"] = originality_vs_ref(M, R)

# diversidade de pool por (sessão × rodada) e por sessão
rows=[]
for (sid,r), g in ideas.groupby(["session_id","round"]):
    rows.append(dict(session_id=sid, round=r, pool_div_round=pool_diversity(np.array([EMB[i] for i in g["idea_id"]]))))
pool_round = pd.DataFrame(rows)
rounds = rounds.merge(pool_round, on=["session_id","round"], how="left")

sess_div=[]
for sid, g in ideas.groupby("session_id"):
    M = np.array([EMB[i] for i in g["idea_id"]])
    sess_div.append(dict(session_id=sid, pool_div=pool_diversity(M), mean_orig=g["orig_score"].mean(),
                         flexibility=flexibility(M)))
sessions = sessions.merge(pd.DataFrame(sess_div), on="session_id", how="left")
display(sessions[["session_id","cell_best_idea","pool_div","mean_orig","flexibility","match_rate"]])

,session_id,cell_best_idea,pool_div,mean_orig,flexibility,match_rate
0,20260701-130722_41ed15ee,judge,0.390730,0.402497,2,0.666667
1,20260701-130823_232719bc,judge,0.445723,0.432246,2,0.333333
2,20260701-130905_73efa694,judge,0.341910,0.384337,5,1.000000
3,20260701-130938_a5480b01,judge,0.428897,0.425346,2,0.666667
4,20260701-131018_63155938,judge,0.424425,0.429326,8,0.666667
...,...,...,...,...,...,...
103,20260701-141557_9502421c,player_decides,0.380951,0.377978,6,1.000000
104,20260701-141646_9752e625,player_decides,0.373275,0.379218,4,0.333333
105,20260701-141725_21ade793,player_decides,0.379580,0.390671,2,0.666667
106,20260701-141804_a990d3a9,player_decides,0.370068,0.367541,3,0.333333


## 6. RQ2 — Homogeneização por dinâmica de decisão (figura‑manchete F2)

**Diversidade coletiva** = diversidade entre as ideias **vencedoras** de sessões
da mesma célula × tema. Menor = mais homogeneizado. Precisa de ≥2 sessões por
célula×tema (se faltar, reporta NaN).

In [9]:
winners = ideas[ideas["advanced"]].copy()
coll=[]
for (mode, tema), g in winners.groupby(["cell_best_idea","cell_tema"]):
    M = np.array([EMB[i] for i in g["idea_id"]])
    coll.append(dict(cell_best_idea=mode, cell_tema=tema, n_sessions=g["session_id"].nunique(),
                     n_winners=len(g), collective_div=pool_diversity(M)))
collective = pd.DataFrame(coll)
display(collective)
collective.to_csv(f"{OUT}/tabelas/T4_diversidade_coletiva.csv", index=False)

if collective["collective_div"].notna().any():
    ax = collective.groupby("cell_best_idea")["collective_div"].mean().plot.bar(
        color="#4f81bd", figsize=(6,4)); ax.set_ylabel("diversidade coletiva (↑ = + diverso)")
    ax.set_title("F2 — Homogeneização por dinâmica de decisão"); ax.set_xlabel("best_idea")
    plt.tight_layout(); plt.savefig(f"{OUT}/figuras/F2_diversidade_coletiva.png", bbox_inches="tight"); plt.show()
else:
    print("Poucas sessões por célula×tema para diversidade coletiva (esperado no piloto).")

,cell_best_idea,cell_tema,n_sessions,n_winners,collective_div
0,judge,Como reduzir as alucinações de modelos de ling...,6,18,0.322114
1,judge,Como reduzir o desperdício de comida em casa?,6,18,0.413926
2,judge,"Como seria uma cidade projetada para o ócio, e...",6,18,0.275302
3,judge,Como uma cidade média pode se adaptar a evento...,6,18,0.371306
4,judge,O que torna uma vida bem vivida?,6,18,0.441287
5,judge,Uma nova forma de as pessoas fazerem amizades ...,6,18,0.303494
6,player,Como reduzir as alucinações de modelos de ling...,6,18,0.327725
7,player,Como reduzir o desperdício de comida em casa?,6,18,0.280834
8,player,"Como seria uma cidade projetada para o ócio, e...",6,18,0.163413
9,player,Como uma cidade média pode se adaptar a evento...,6,18,0.269828


## 7. RQ4 — Trajetória de convergência (F3)

Diversidade do pool por rodada e *slope* por sessão. Inclinação negativa = pool
fica mais homogêneo ao longo do loop.

In [10]:
from numpy.polynomial import polynomial as Poly
slopes=[]
for sid, g in rounds.dropna(subset=["pool_div_round"]).groupby("session_id"):
    if g["round"].nunique() >= 2:
        b = np.polyfit(g["round"], g["pool_div_round"], 1)[0]
        slopes.append(dict(session_id=sid, conv_slope=b))
slopes = pd.DataFrame(slopes)
if len(slopes): sessions = sessions.merge(slopes, on="session_id", how="left")

traj = rounds.groupby(["cell_best_idea","round"])["pool_div_round"].mean().reset_index()
if traj["pool_div_round"].notna().any():
    fig, ax = plt.subplots(figsize=(7,4))
    for mode, g in traj.groupby("cell_best_idea"):
        ax.plot(g["round"], g["pool_div_round"], marker="o", label=mode)
    ax.set_xlabel("rodada"); ax.set_ylabel("diversidade do pool"); ax.legend(title="best_idea")
    ax.set_title("F3 — Trajetória de diversidade por rodada")
    plt.tight_layout(); plt.savefig(f"{OUT}/figuras/F3_convergencia.png", bbox_inches="tight"); plt.show()
if len(slopes): display(slopes.merge(sessions[["session_id","cell_best_idea"]], on="session_id"))

,session_id,conv_slope,cell_best_idea
0,20260701-130722_41ed15ee,0.013533,judge
1,20260701-130823_232719bc,0.001601,judge
2,20260701-130905_73efa694,-0.012044,judge
3,20260701-130938_a5480b01,-0.029221,judge
4,20260701-131018_63155938,0.031153,judge
...,...,...,...
103,20260701-141557_9502421c,0.005716,player_decides
104,20260701-141646_9752e625,0.006107,player_decides
105,20260701-141725_21ade793,-0.006575,player_decides
106,20260701-141804_a990d3a9,-0.068539,player_decides


## 8. Concordância juiz–jogador ao longo das rodadas (F6)

`match` por rodada (jogador e juiz escolheram a mesma ideia). Tendência crescente
sugere deferência do jogador‑agente ao juiz.

In [11]:
agree = rounds.groupby(["cell_best_idea","round"])["match"].mean().reset_index()
if agree["match"].notna().any():
    fig, ax = plt.subplots(figsize=(7,4))
    for mode, g in agree.groupby("cell_best_idea"):
        ax.plot(g["round"], g["match"], marker="s", label=mode)
    ax.set_ylim(-0.05,1.05); ax.set_xlabel("rodada"); ax.set_ylabel("P(jogador = juiz)")
    ax.set_title("F6 — Concordância juiz–jogador por rodada"); ax.legend(title="best_idea")
    plt.tight_layout(); plt.savefig(f"{OUT}/figuras/F6_concordancia.png", bbox_inches="tight"); plt.show()
print("match médio por dinâmica:")
display(rounds.groupby("cell_best_idea")["match"].mean().rename("match_rate").reset_index())

match médio por dinâmica:


,cell_best_idea,match_rate
0,judge,0.601852
1,player,0.648148
2,player_decides,0.601852


## 9. Autoria dos vencedores

Fração de ideias que avançam por papel (jogador‑agente vs personas) e por dinâmica.

In [12]:
auth = (winners.groupby(["cell_best_idea","author_role"]).size()
        .rename("n").reset_index())
piv = auth.pivot(index="cell_best_idea", columns="author_role", values="n").fillna(0)
display(piv)
ax = piv.plot.bar(stacked=True, figsize=(7,4)); ax.set_ylabel("nº de vencedores")
ax.set_title("Autoria dos vencedores por dinâmica"); ax.set_xlabel("best_idea")
plt.tight_layout(); plt.savefig(f"{OUT}/figuras/autoria_vencedores.png", bbox_inches="tight"); plt.show()

author_role,persona,player
cell_best_idea,,
judge,44,64
player,7,101
player_decides,6,102


## 10. RQ3 — Auto‑preferência / sensibilidade à fonte

Δ_self = P(escolher autoria própria | identificada) − P(... | anônima), por escolhedor.
Só é estimável quando a cegueira de fonte **varia** no run.

In [13]:
if sessions["cell_source_blind"].nunique() >= 2:
    # juiz: "próprio" = mesma família de modelo (cross_model) ou heurística a definir
    sub = ideas.copy()
    sub["judge_pick"] = sub["selected_by_judge"].astype(int)
    g = sub.groupby("cell_source_blind")["judge_pick"].mean()
    print("Taxa de seleção pelo juiz por condição de cegueira:"); display(g)
    print("Δ_self (juiz) =", round(g.get(False,np.nan)-g.get(True,np.nan), 4),
          "  (placeholder: definir 'autoria própria' p/ cross_model)")
else:
    print("Cegueira de fonte é fixa neste run (", sessions["cell_source_blind"].unique(),
          ") → Δ_self não estimável. Infra pronta para quando C variar.")

Cegueira de fonte é fixa neste run ( [ True] ) → Δ_self não estimável. Infra pronta para quando C variar.


## 11. Efeito da síntese

A síntese final do Dono do Bar amplifica ou mitiga a homogeneização? Compara a
síntese com o pool de ideias da sessão (no espaço de embedding).

In [14]:
syn_rows=[]
for sid, g in ideas.groupby("session_id"):
    es = SYN_EMB.get(sid)
    if es is None or float(np.linalg.norm(es)) == 0: continue   # sem síntese
    Es = np.asarray(es).reshape(1, -1); M = np.array([EMB[i] for i in g["idea_id"]])
    o = float(originality_vs_ref(Es, M)[0])   # originalidade da síntese vs pool da sessão
    syn_rows.append(dict(session_id=sid, synth_orig_vs_pool=o, pool_div=pool_diversity(M)))
syn = pd.DataFrame(syn_rows)
if len(syn):
    sessions = sessions.merge(syn[["session_id","synth_orig_vs_pool"]], on="session_id", how="left")
    display(syn.describe()[["synth_orig_vs_pool","pool_div"]])

,synth_orig_vs_pool,pool_div
count,108.000000,108.000000
mean,0.453806,0.426334
std,0.060866,0.046230
min,0.358662,0.332809
25%,0.413808,0.395360
50%,0.440863,0.427979
75%,0.477539,0.454818
max,0.622872,0.553746


## 12. Modelos estatísticos (com *guards* para N pequeno)

No piloto a única variação é `best_idea`. Onde houver ≥2 níveis e dados suficientes,
ajusta um modelo; senão, reporta descritivo. No estudo final: LMM com interceptos
aleatórios por jogador e tema.

In [15]:
import statsmodels.formula.api as smf
def try_anova(df, dv, factor):
    sub = df.dropna(subset=[dv]).copy()
    if sub[factor].nunique() < 2 or len(sub) < sub[factor].nunique()+2:
        print(f"[{dv} ~ {factor}] dados insuficientes (piloto)."); return
    try:
        m = smf.ols(f"{dv} ~ C({factor})", data=sub).fit()
        print(f"=== {dv} ~ {factor} ===")
        print(m.summary().tables[1]); print("R²:", round(m.rsquared,3), "\n")
    except Exception as e:
        print(f"[{dv} ~ {factor}] falhou: {e}")

for dv in ["pool_div","mean_orig","flexibility","conv_slope"]:
    if dv in sessions.columns: try_anova(sessions, dv, "cell_best_idea")

=== pool_div ~ cell_best_idea ===
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
Intercept                               0.4385      0.008     57.408      0.000       0.423       0.454
C(cell_best_idea)[T.player]            -0.0201      0.011     -1.858      0.066      -0.041       0.001
C(cell_best_idea)[T.player_decides]    -0.0163      0.011     -1.508      0.134      -0.038       0.005
R²: 0.036 

=== mean_orig ~ cell_best_idea ===
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
Intercept                               0.4490      0.008     58.536      0.000       0.434       0.464
C(cell_best_idea)[T.player]            -0.0079      0.011     -0.728      0.468      -0

## 13. Análise qualitativa de conteúdo

Lê o conteúdo: cadeia de vencedores por sessão (como a ideia evolui), exemplos
lado a lado, e mapa 2D dos embeddings colorido por dinâmica/rodada (F7).

In [16]:
# Cadeia de vencedores de uma sessão exemplo
ex = sessions["session_id"].iloc[0]
print(f"Cadeia de vencedores — sessão {ex} (modo {sessions['cell_best_idea'].iloc[0]}):\n")
for _, w in winners[winners["session_id"]==ex].sort_values("round").iterrows():
    print(f"  Rodada {w['round']} — venceu {w['author_role']} ({w['author_name']}):")
    print(f"     {w['text'][:160]}...\n")

# Mapa 2D (F7): PCA dos embeddings (UMAP se disponível)
try:
    import umap; XY = umap.UMAP(n_neighbors=min(10,len(E)-1), random_state=0).fit_transform(E); méth="UMAP"
except Exception:
    from sklearn.decomposition import PCA; XY = PCA(2, random_state=0).fit_transform(np.asarray(E)); méth="PCA"
fig, ax = plt.subplots(1,2, figsize=(13,5))
for mode, idx in ideas.groupby("cell_best_idea").groups.items():
    ii=list(idx); ax[0].scatter(XY[ii,0], XY[ii,1], label=str(mode), alpha=.7)
ax[0].set_title(f"F7 — ideias no espaço ({méth}), por dinâmica"); ax[0].legend(fontsize=8)
sc = ax[1].scatter(XY[:,0], XY[:,1], c=ideas["round"], cmap="viridis", alpha=.8)
ax[1].set_title("ideias por rodada"); plt.colorbar(sc, ax=ax[1], label="rodada")
plt.tight_layout(); plt.savefig(f"{OUT}/figuras/F7_embeddings_2d.png", bbox_inches="tight"); plt.show()

Cadeia de vencedores — sessão 20260701-130722_41ed15ee (modo judge):

  Rodada 1 — venceu player (Jogador Cauteloso (T=0.7)):
     Eu posso criar um planejamento semanal de refeições e fazer uma lista de compras baseada nisso, garantindo que só compre o necessário e utilizando os ingredient...

  Rodada 2 — venceu persona (Bia):
     Planejamento é tudo, mas que tal usar apps de receitas que trabalham com o que temos em casa? Assim, a gente não só evita o desperdício, mas ainda dá uma animad...

  Rodada 3 — venceu player (Jogador Cauteloso (T=0.7)):
     Eu proponho que cada família crie um cardápio semanal que utilize todos os ingredientes que já têm em casa, planejando as refeições para evitar compras excessiv...



## 14. Sumário executivo (automático)

In [17]:
print("="*64)
print("SUMÁRIO — run:", DATA_ROOT)
print("="*64)
print(f"Sessões: {len(sessions)} | ideias: {len(ideas)} | chamadas: {len(calls)}")
print(f"Custo total estimado: US$ {calls['cost_usd'].sum():.4f} "
      f"(US$ {calls['cost_usd'].sum()/max(len(sessions),1):.5f}/sessão)")
print(f"Tokens: {int(calls['tokens_in'].sum())} in / {int(calls['tokens_out'].sum())} out")
print(f"Falhas: {int(calls['error'].sum())} | parse ok: {100*calls['parse_status'].eq('ok').mean():.1f}%")
print(f"Backend de embedding: {EMB_BACKEND}")
if "pool_div" in sessions:
    print("\nDiversidade de pool média por dinâmica:")
    print(sessions.groupby('cell_best_idea')['pool_div'].mean().round(4).to_string())
if "conv_slope" in sessions and sessions["conv_slope"].notna().any():
    print("\nSlope de convergência médio (negativo = homogeneíza):",
          round(sessions["conv_slope"].mean(),4))
print("\nAutoria dos vencedores:")
print(winners["author_role"].value_counts().to_string())

SUMÁRIO — run: ../data/sim_runs/sim_20260701_130722
Sessões: 108 | ideias: 1620 | chamadas: 3672
Custo total estimado: US$ 0.3976 (US$ 0.00368/sessão)
Tokens: 1728222 in / 230589 out
Falhas: 0 | parse ok: 100.0%
Backend de embedding: st:paraphrase-multilingual-mpnet-base-v2

Diversidade de pool média por dinâmica:
cell_best_idea
judge             0.4385
player            0.4184
player_decides    0.4222

Slope de convergência médio (negativo = homogeneíza): -0.0155

Autoria dos vencedores:
author_role
player     267
persona     57


## 15. Exportações

Salva as tabelas *tidy* (CSV) e deixa as figuras em `analise_saidas/figuras/`.

In [18]:
sessions.to_csv(f"{OUT}/tabelas/sessions.csv", index=False)
ideas.drop(columns=["text"]).to_csv(f"{OUT}/tabelas/ideas.csv", index=False)
rounds.to_csv(f"{OUT}/tabelas/rounds.csv", index=False)
calls.to_csv(f"{OUT}/tabelas/llm_calls.csv", index=False)
print("Exportado para", OUT)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f): print("  ", f)

Exportado para analise_saidas
   analise_saidas/figuras/F2_diversidade_coletiva.png
   analise_saidas/figuras/F3_convergencia.png
   analise_saidas/figuras/F6_concordancia.png
   analise_saidas/figuras/F7_embeddings_2d.png
   analise_saidas/figuras/autoria_vencedores.png
   analise_saidas/figuras/eng_tokens_custo_latencia.png
   analise_saidas/tabelas/T4_diversidade_coletiva.csv
   analise_saidas/tabelas/T7_confiabilidade.csv
   analise_saidas/tabelas/ideas.csv
   analise_saidas/tabelas/llm_calls.csv
   analise_saidas/tabelas/rounds.csv
   analise_saidas/tabelas/sessions.csv
